# はじめに

このノートではいまさらながらニューラルネットワークの数理についてまとめておく。将来的に競艇の予測モデルをディープラーニングを使って行いたく、その前に基本的な数理の部分、とくにバックプロパゲーション法をおさらいしておく。


## ニューラルネットワークの数理


<div align="center"><img src="./deeplearning1.png" width="1200"></div>

<div align="center"><img src="./deeplearning2.png" width="1200"></div>

## ニューラルネットワークを作る

In [4]:
import numpy as np


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def sigmoid_deriv(a):
    # a = sigmoid(x)
    return a * (1 - a)


class SimpleNN:
    def __init__(self, in_dim, hid_dim, out_dim, lr=0.1, seed=0):
        np.random.seed(seed)

        # 重み
        self.W1 = np.random.randn(in_dim, hid_dim)
        self.b1 = np.random.rand(hid_dim)

        self.W2 = np.random.randn(hid_dim, out_dim)
        self.b2 = np.random.rand(out_dim)

        self.lr = lr

    def forward(self, X):
        # ->隠れ層
        self.z1 = X @ self.W1 + self.b1
        self.a1 = sigmoid(self.z1)

        # ->出力層
        self.z2 = self.a1 @ self.W2 + self.b2
        self.y_hat = sigmoid(self.z2)

        return self.y_hat

    def backward(self, X, y):
        n = X.shape[0]

        # <-出力層デルタ
        error = y - self.y_hat
        delta2 = error * sigmoid_deriv(self.y_hat)

        # <-隠れ層デルタ
        delta1 = (delta2 @ self.W2.T) * sigmoid_deriv(self.a1)

        # 重み更新
        self.W2 += self.lr * self.a1.T @ delta2
        self.b2 += self.lr * delta2.sum(axis=0)

        self.W1 += self.lr * X.T @ delta1
        self.b1 += self.lr * delta1.sum(axis=0)

    def fit(self, X, y, epochs=10000):
        for _ in range(epochs):
            self.forward(X)
            self.backward(X, y)

    def predict(self, X):
        return self.forward(X)

In [5]:
# データの設定
np.random.seed(1)
n_samples = 100  # サンプル数
in_dim = 2       # 入力次元（特徴量の数）
hid_dim = 4      # 隠れ層の次元
out_dim = 1      # 出力次元（二値分類の場合は1）

# 入力データX: (n_samples, in_dim)
# 2つの特徴量を持つデータを生成
X = np.random.randn(n_samples, in_dim)

# ターゲットデータy: (n_samples, out_dim)
y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(float).reshape(-1, 1)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"X (最初の5サンプル):\n{X[:5]}")
print(f"y (最初の5サンプル):\n{y[:5]}")

X shape: (100, 2)
y shape: (100, 1)
X (最初の5サンプル):
[[ 1.62434536 -0.61175641]
 [-0.52817175 -1.07296862]
 [ 0.86540763 -2.3015387 ]
 [ 1.74481176 -0.7612069 ]
 [ 0.3190391  -0.24937038]]
y (最初の5サンプル):
[[1.]
 [0.]
 [1.]
 [1.]
 [1.]]


In [6]:
# ニューラルネットワークの初期化と学習
snn = SimpleNN(in_dim=in_dim, hid_dim=hid_dim, out_dim=out_dim, lr=0.1, seed=42)

# 学習
print("学習開始...")
snn.fit(X, y, epochs=10000)
print("学習完了")

# 予測
predictions = snn.predict(X)
print(f"\n予測結果の形状: {predictions.shape}")
print(f"予測値 (最初の10サンプル):\n{predictions[:10].flatten()}")
print(f"正解値 (最初の10サンプル):\n{y[:10].flatten()}")

# 予測精度の確認（二値分類として）
predicted_classes = (predictions > 0.5).astype(int)
accuracy = np.mean(predicted_classes == y)
print(f"\n精度: {accuracy:.4f}")

学習開始...
学習完了

予測結果の形状: (100, 1)
予測値 (最初の10サンプル):
[9.89272524e-01 4.57724732e-08 9.98914736e-01 9.93815906e-01
 9.95830955e-01 9.99999999e-01 2.90148123e-03 1.00000000e+00
 4.84001044e-03 1.12308180e-01]
正解値 (最初の10サンプル):
[1. 0. 1. 1. 1. 1. 0. 1. 0. 0.]

精度: 0.9900
